# Notebook 06 — Int16-Safe Deployment Pipeline

**Project:** AIoT Predictive Maintenance for Semiconductor Test Equipment  
**Module:** LD7182 — AI for IoT  
**Author:** Rekhanth Reddy Obireddy  
**Date:** 5 May 2026

## Background

Notebook 05 documented an attempt to deploy a Random Forest classifier to ESP32 via emlearn's inline C export. The deployment failed because emlearn's `(int16_t)values[i]` cast destroyed information from z-score standardised features (most values fall in [-3, +3] range, truncated to small integers).

## This Notebook's Approach

Re-engineer the preprocessing to use min-max scaling to an int16-safe range (-30000 to +30000) instead of StandardScaler. This makes emlearn's float-to-int16 cast lossless for our purposes (rounding error < 1.0 unit per feature).

## Outcome

Random Forest deployed to ESP32 with verified predictions matching sklearn reference. All 5 hardcoded SECOM test samples (3 Pass + 2 Fail) predict correctly. End-to-end IoT pipeline operational.

In [1]:
# Setup and imports
!pip install emlearn -q

import numpy as np
import pandas as pd
import emlearn
import joblib
import os
import inspect
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, confusion_matrix
)

RANDOM_STATE = 42
print("✅ Imports complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 5.0 MB/s eta 0:00:00
✅ Imports complete


In [2]:
# Load raw SECOM data (must be uploaded to Colab files)
print("Loading raw SECOM data...")
X_raw = pd.read_csv('secom.data', sep=r'\s+', header=None)
y_raw = pd.read_csv('secom_labels.data', sep=r'\s+', header=None)

y_labels = y_raw[0].values
y_binary = (y_labels == 1).astype(int)

print(f"Shape: {X_raw.shape}")
print(f"Pass: {(y_binary == 0).sum()}, Fail: {(y_binary == 1).sum()}")

Loading raw SECOM data...
Shape: (1567, 590)
Pass: 1463, Fail: 104


In [3]:
# Drop high-missing features (>55%)
missing_pct = X_raw.isnull().mean()
high_missing_cols = missing_pct[missing_pct > 0.55].index.tolist()
X_step1 = X_raw.drop(columns=high_missing_cols)
print(f"After dropping high-missing: {X_step1.shape}")

# Drop zero/near-zero variance features
imputer_temp = SimpleImputer(strategy='median')
X_imputed_temp = pd.DataFrame(imputer_temp.fit_transform(X_step1), columns=X_step1.columns)
variances = X_imputed_temp.var()
low_var_cols = variances[variances < 1e-6].index.tolist()
X_step2 = X_step1.drop(columns=low_var_cols)
print(f"After dropping low-variance: {X_step2.shape}")

# Median imputation only — no StandardScaler
imputer_final = SimpleImputer(strategy='median')
X_unscaled = imputer_final.fit_transform(X_step2)
print(f"Final unscaled shape: {X_unscaled.shape}")
print(f"Value range: [{X_unscaled.min():.2f}, {X_unscaled.max():.2f}]")

After dropping high-missing: (1567, 566)
After dropping low-variance: (1567, 440)
Final unscaled shape: (1567, 440)
Value range: [-14804.50, 37943.00]


In [4]:
# The critical fix: scale to a range that survives int16 cast
INT16_SAFE_RANGE = (-30000, 30000)
minmax_scaler = MinMaxScaler(feature_range=INT16_SAFE_RANGE)
X_int16safe = minmax_scaler.fit_transform(X_unscaled)

# Verify int16 cast preserves values
X_int16_cast = X_int16safe.astype(np.int16).astype(np.float32)
roundtrip_diff = np.abs(X_int16safe - X_int16_cast).max()

print(f"Scaled range: [{X_int16safe.min():.2f}, {X_int16safe.max():.2f}]")
print(f"Max int16 cast roundtrip diff: {roundtrip_diff:.2f}")
print("(Should be < 1.0 — rounding error only)")

joblib.dump(minmax_scaler, 'minmax_scaler_int16.joblib')

Scaled range: [-30000.00, 30000.00]
Max int16 cast roundtrip diff: 1.00
(Should be < 1.0 — rounding error only)


['minmax_scaler_int16.joblib']

In [5]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_int16safe, y_binary,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_binary
)

# Train RF
rf_int16 = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_int16.fit(X_train, y_train)

# Threshold tuning
proba = rf_int16.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.05, 0.96, 0.01)
f1s = [f1_score(y_test, (proba >= t).astype(int)) if (proba >= t).sum() > 0 else 0 for t in thresholds]
best_t = thresholds[np.argmax(f1s)]
best_f1 = max(f1s)

print(f"\nBest threshold: {best_t:.2f}")
print(f"F1: {best_f1:.4f}")
print(f"AUC: {roc_auc_score(y_test, proba):.4f}")


Best threshold: 0.19
F1: 0.3590
AUC: 0.7432


In [6]:
# Generate C deployment file
cmodel = emlearn.convert(rf_int16, method='inline')
cmodel.save(file='rf_secom_int16safe.h', name='rf_secom_i16')

size_kb = os.path.getsize('rf_secom_int16safe.h') / 1024
print(f"Generated: rf_secom_int16safe.h ({size_kb:.1f} KB)")

# Verify int16 cast preserves predictions
X_test_cast = X_test.astype(np.int16).astype(np.float32)
proba_original = rf_int16.predict_proba(X_test)[:, 1]
proba_cast = rf_int16.predict_proba(X_test_cast)[:, 1]
max_diff = np.abs(proba_original - proba_cast).max()

agreement = ((proba_original >= best_t).astype(int) == (proba_cast >= best_t).astype(int)).mean() * 100
print(f"Max prob diff: {max_diff:.6f}")
print(f"Prediction agreement at threshold {best_t:.2f}: {agreement:.2f}%")

Generated: rf_secom_int16safe.h (1542.3 KB)
Max prob diff: 0.019432
Prediction agreement at threshold 0.19: 99.68%


In [7]:
# Pick 5 representative samples (3 Pass + 2 Fail)
pass_indices = np.where(y_test == 0)[0]
fail_indices = np.where(y_test == 1)[0]
top_pass = pass_indices[np.argsort(proba[pass_indices])[:3]]
top_fail = fail_indices[np.argsort(proba[fail_indices])[-2:][::-1]]

selected = list(top_pass) + list(top_fail)
labels = [0]*3 + [1]*2

# Write secom_samples.h
with open('secom_samples.h', 'w') as f:
    f.write("// Stored SECOM test samples for ESP32 inference\n")
    f.write(f"#define NUM_SAMPLES {len(selected)}\n")
    f.write(f"#define N_FEATURES {X_test.shape[1]}\n\n")

    for i, idx in enumerate(selected):
        sample = X_test[idx].astype(np.int16)
        label = "PASS" if labels[i] == 0 else "FAIL"
        f.write(f"// Sample {i}: {label} (sklearn prob_fail={proba[idx]:.4f})\n")
        f.write(f"const int16_t SAMPLE_{i}[N_FEATURES] = {{\n")
        for j in range(0, len(sample), 10):
            chunk = ", ".join(f"{v:6d}" for v in sample[j:j+10])
            sep = "," if j + 10 < len(sample) else ""
            f.write(f"    {chunk}{sep}\n")
        f.write("};\n\n")

    f.write("const int16_t* SAMPLES[NUM_SAMPLES] = {\n")
    for i in range(len(selected)):
        sep = "," if i < len(selected)-1 else ""
        f.write(f"    SAMPLE_{i}{sep}\n")
    f.write("};\n\n")

    f.write("const int SAMPLE_LABELS[NUM_SAMPLES] = {")
    f.write(", ".join(str(l) for l in labels))
    f.write("};\n\n")

    f.write("const char* SAMPLE_NAMES[NUM_SAMPLES] = {\n")
    for i, lbl in enumerate(labels):
        label_str = "Pass" if lbl == 0 else "Fail"
        sep = "," if i < len(selected)-1 else ""
        f.write(f'    "Sample {i} ({label_str})"{sep}\n')
    f.write("};\n")

print(f"✅ Generated secom_samples.h ({os.path.getsize('secom_samples.h')} bytes)")
print(f"\n=== DEPLOYMENT FILES READY ===")
print(f"  rf_secom_int16safe.h — model")
print(f"  secom_samples.h — test data")
print(f"  Threshold to use: {best_t:.2f}")

✅ Generated secom_samples.h (19339 bytes)

=== DEPLOYMENT FILES READY ===
  rf_secom_int16safe.h — model
  secom_samples.h — test data
  Threshold to use: 0.19
